# Saved API response and enrichment limits

**Partial API workflow.** The real IBM JSON snapshot is parsed and filtered locally. Rocket, payload, core and launchpad enrichment is not reconstructed. The default run makes no network requests; the provided reference CSV is not presented as a newly recreated API result.

Run the cells in order from the repository root or `notebooks/`. Install `requirements.txt` and run `download_inputs.py` if `data/` is missing. Local output files are written under `results/`.

## Sources and credit

Adapted from this repository's `scripts/06_api_data.py` and the public IBM DS0321EN Skills Network SpaceX labs. IBM and the credited lab authors, including Joseph Santarcangelo, Nayef Abou Tayoun and Yan Luo, retain their source credit. Input URLs and SHA-256 hashes are listed in `SOURCES.json`; see `ATTRIBUTION.md` for third-party terms. This notebook does not grant a new licence over third-party material.


In [1]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import pandas as pd
pd.set_option('display.max_rows', 12)
pd.set_option('display.max_columns', 12)
pd.set_option('display.max_colwidth', 70)

In [2]:
from pathlib import Path

import os

import matplotlib

matplotlib.use('module://matplotlib_inline.backend_inline')

from IPython.display import display

HERE = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data').is_dir() and (p / 'SOURCES.json').is_file()), None)

if HERE is None:
    raise FileNotFoundError('Open this notebook from the repository root or its notebooks folder; run download_inputs.py first.')

DATA = HERE / 'data'

OUT = HERE / 'results' / '06_api_partial'

OUT.mkdir(parents=True, exist_ok=True)

os.chdir(OUT)

import argparse, datetime, json

import pandas as pd

import requests

parser = argparse.ArgumentParser(description='Parse the exact IBM static launch snapshot; no enrichment is fabricated.')

parser.add_argument('--probe-live', action='store_true', help='Optional single GET with 10-second timeout; no retries or per-record enrichment')

args = parser.parse_args([])

live_available = False

probe = {'requested': args.probe_live, 'method': 'default local snapshot only', 'enrichment_completed': False}

In [3]:
if args.probe_live:
    try:
        response = requests.get('https://api.spacexdata.com/v4/launches/past', timeout=10)
        probe['http_status'] = response.status_code
        live_available = response.status_code == 200
    except requests.RequestException as exc:
        probe['error'] = repr(exc)

(OUT / 'LIVE_PROBE.json').write_text(json.dumps(probe, indent=2), encoding='utf-8')

static_json_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'

import json, hashlib

static_path = DATA / 'API_call_spacex_api.json'

static_payload = json.loads(static_path.read_text())

acquisition = next((r for r in json.loads((HERE / 'DATA_ACQUISITION.json').read_text()) if Path(r['path']).name == static_path.name))

assert hashlib.sha256(static_path.read_bytes()).hexdigest() == acquisition['sha256']

{k: acquisition[k] for k in ['url', 'status', 'bytes', 'sha256']}

data = pd.json_normalize(static_payload)

data.head(5)

data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]

data = data[data['cores'].map(len) == 1]

data = data[data['payloads'].map(len) == 1]

In [4]:
data['cores'] = data['cores'].map(lambda x: x[0])

data['payloads'] = data['payloads'].map(lambda x: x[0])

data['date'] = pd.to_datetime(data['date_utc']).dt.date

data = data[data['date'] <= datetime.date(2020, 11, 13)]

BoosterVersion = []

PayloadMass = []

Orbit = []

LaunchSite = []

Outcome = []

Flights = []

GridFins = []

Reused = []

Legs = []

LandingPad = []

Block = []

ReusedCount = []

Serial = []

Longitude = []

Latitude = []

BoosterVersion

[]

In [5]:
data.to_json(OUT / 'static_launch_subset.json', orient='records', date_format='iso')

(OUT / 'PARTIAL.json').write_text(json.dumps({'static_launch_rows_after_template_filters': len(data), 'live_available': live_available, 'blocked': 'Rocket, launchpad, payload and core enrichment endpoints unavailable; official dataset_part_1 is not claimed as a recreated API result.'}, indent=2))

print('PARTIAL: static snapshot parsed; live enrichment was not reconstructed.')

PARTIAL: static snapshot parsed; live enrichment was not reconstructed.
